In [ ]:
import requests, random, os, time, threading
from svglib.svglib import svg2rlg
from reportlab.graphics import renderPM
import io

In [ ]:
# Create output directory for storing collected captcha images
os.makedirs("./data_collection", exist_ok=True)

In [ ]:
def data(idx):
    """
    Download a single captcha image from the server and save it as PNG.
    """
    url = "http://218.5.5.242:9019/captcha"
    params = {'_': random.random()}  # Random parameter to bypass cache
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status() 
    svg_content = response.content

    # Convert SVG to PNG and save
    file_path = os.path.join("./data_collection", f"img{idx+1}.png")

    svg_file = io.BytesIO(svg_content)
    drawing = svg2rlg(svg_file)
    renderPM.drawToFile(drawing, file_path, fmt="PNG")


def collection(start, end):
    """
    Collect captcha images in batch, skipping already downloaded ones.
    """
    for idx in range(start, end):
        if f"img{idx+1}.png" in os.listdir("data_collection"):
            continue  # Skip if already exists
        data(idx)
        if not idx % 500:
            print(f"It has collected {idx+1} images")

In [ ]:
# Multi-threaded collection: 10 workers, each collecting 500 images
workflow = 10
amount = 500 

threads = []
for i in range(workflow):
    t = threading.Thread(target=collection, args=(i*amount, (i+1)*amount))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

In [ ]:
print(len(os.listdir("data_collection")))